In [0]:
-- 1. Clean & Deduplicate Rentals (Silver)
CREATE OR REPLACE TABLE silver.rentals_clean
USING DELTA
AS
WITH parsed_rentals AS (
  SELECT
    _id[0] AS rental_id,
    INITCAP(TRIM(city)) AS city,
    UPPER(REGEXP_REPLACE(TRIM(postalCode), '\\s+', '')) AS raw_pc6,
    REGEXP_EXTRACT(UPPER(TRIM(postalCode)), '^([1-9][0-9]{3}\\s*[A-Z]{2})$', 1) AS matched_pc6,
    TRIM(propertyType) AS property_type,
    CASE
      WHEN propertyType IN ('Apartment', 'Studio') THEN 'Entire home/apt'
      WHEN propertyType = 'Room' THEN 'Private room'
      ELSE 'Other'
    END AS standardized_room_type,
    CAST(REGEXP_EXTRACT(rent, '([0-9]+(?:[.,][0-9]+)?)', 1) AS DOUBLE) AS base_monthly_rent,
    CASE
      WHEN REGEXP_EXTRACT(additionalCostsRaw, '([0-9]+(?:[.,][0-9]+)?)', 1) = '' THEN 0.0
      ELSE CAST(REGEXP_EXTRACT(additionalCostsRaw, '([0-9]+(?:[.,][0-9]+)?)', 1) AS DOUBLE)
    END AS additional_monthly_costs,
    REGEXP_EXTRACT(additionalCostsRaw, '([0-9]+(?:[.,][0-9]+)?)', 1) != '' AS has_additional_costs,
    CASE WHEN rent ilike '(%utilities incl%)' THEN TRUE ELSE FALSE END AS is_utilities_included,
    CAST(REGEXP_EXTRACT(areaSqm, '([0-9]+)', 1) AS INT) AS area_sqm,
    CAST(latitude AS decimal(28,8)) AS latitude,
    CAST(longitude AS decimal(28,8)) AS longitude,
    furnish,
    energyLabel AS energy_label,
    isRoomActive AS is_room_active,
    TRY_TO_TIMESTAMP(lastSeenAt[0]) AS last_seen_at,
    _ingested_at
  FROM bronze.rentals_raw
  WHERE crawlStatus = 'done'
),
validated_rentals AS (
  SELECT
    *,
    REGEXP_REPLACE(matched_pc6, '\\s+', '') AS pc6,
    SUBSTRING(REGEXP_REPLACE(matched_pc6, '\\s+', ''), 1, 4) AS pc4,
    CASE WHEN matched_pc6 IS NOT NULL AND matched_pc6 != '' THEN TRUE ELSE FALSE END AS is_valid_zipcode,
    (base_monthly_rent + additional_monthly_costs) AS total_monthly_rent,
    ROW_NUMBER() OVER (
      PARTITION BY raw_pc6, ROUND(latitude, 5), ROUND(longitude, 5), property_type, base_monthly_rent, area_sqm
      ORDER BY last_seen_at DESC NULLS LAST, rental_id ASC
    ) AS rn
  FROM parsed_rentals
  WHERE base_monthly_rent IS NOT NULL
    AND base_monthly_rent > 0
)
SELECT
  rental_id,
  city,
  CASE WHEN city ilike ('amsterdam') THEN TRUE ELSE FALSE END AS is_amsterdam,
  pc4,
  pc6,
  is_valid_zipcode,
  property_type,
  standardized_room_type,
  base_monthly_rent,
  additional_monthly_costs,
  has_additional_costs,
  total_monthly_rent,
  is_utilities_included,
  area_sqm,
  latitude,
  longitude,
  furnish,
  energy_label,
  is_room_active,
  last_seen_at,
  _ingested_at
FROM validated_rentals
WHERE rn = 1;

In [0]:
CREATE OR REPLACE TABLE silver.airbnb_clean
USING DELTA
AS
WITH parsed_airbnb AS (
  SELECT
    TRIM(zipcode) AS raw_zipcode,
    REGEXP_EXTRACT(UPPER(TRIM(zipcode)), '([1-9][0-9]{3}\\s*[A-Z]{2}|[1-9][0-9]{3})', 1) AS extracted_zip,
    CAST(latitude AS decimal(28,8)) AS latitude,
    CAST(longitude AS decimal(28,8)) AS longitude,
    TRIM(room_type) AS room_type,
    CAST(accommodates AS INT) AS accommodates,
    CAST(bedrooms AS DOUBLE) AS bedrooms,
    CAST(price AS DOUBLE) AS price,
    CAST(review_scores_value AS DOUBLE) AS review_scores_value,
    _ingested_at
  FROM bronze.airbnb_raw
  WHERE price RLIKE '^[0-9]+(\\.[0-9]+)?$'
    AND CAST(price AS DOUBLE) > 0
),
deduped_airbnb AS (
  SELECT
    *,
    REGEXP_REPLACE(extracted_zip, '\\s+', '') AS clean_zip,
    CASE
      WHEN review_scores_value >= 90 THEN TRUE
      ELSE FALSE
    END AS has_recommendation,
    CASE
      WHEN review_scores_value IS NOT NULL THEN TRUE
      ELSE FALSE
    END AS has_review_score,
    ROW_NUMBER() OVER (
      PARTITION BY ROUND(latitude, 5), ROUND(longitude, 5), room_type, price, accommodates, bedrooms
      ORDER BY review_scores_value DESC NULLS LAST, extracted_zip DESC NULLS LAST
    ) AS rn
  FROM parsed_airbnb
),
base_silver AS (
  SELECT
    MONOTONICALLY_INCREASING_ID() AS airbnb_row_id,
    raw_zipcode,
    NULLIF(SUBSTRING(clean_zip, 1, 4), '') AS parsed_pc4,
    CASE WHEN LENGTH(clean_zip) = 6 THEN clean_zip ELSE NULL END AS pc6,
    latitude,
    longitude,
    CASE
      WHEN latitude IS NOT NULL AND longitude IS NOT NULL
      THEN ST_SetSRID(ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE)), 4326)
      ELSE NULL
    END AS listing_point,
    room_type,
    accommodates,
    bedrooms,
    price,
    review_scores_value,
    has_recommendation,
    has_review_score,
    _ingested_at
  FROM deduped_airbnb
  WHERE rn = 1
),
amsterdam_postcodes AS (
  SELECT
    pc4_code,
    geom_4326,
    ST_XMin(geom_4326) AS min_lon,
    ST_XMax(geom_4326) AS max_lon,
    ST_YMin(geom_4326) AS min_lat,
    ST_YMax(geom_4326) AS max_lat
  FROM (
    SELECT
      pc4_code,
      ST_SetSRID(ST_GeomFromGeoJSON(
        REGEXP_REPLACE(
          REGEXP_REPLACE(raw_geometry_json, '"(?=[\\[0-9-])', ''),
          '(?<=[\\]0-9])"', ''
        )
      ), 4326) AS geom_4326
    FROM bronze.post_codes_raw
    WHERE gem_name = 'Amsterdam'
  )
),
spatially_enriched AS (
  SELECT
    b.*,
    p.pc4_code AS spatial_pc4,
    ROW_NUMBER() OVER (
      PARTITION BY b.airbnb_row_id
      ORDER BY p.pc4_code ASC NULLS LAST
    ) AS spatial_rn
  FROM base_silver b
  LEFT JOIN amsterdam_postcodes p
    ON b.parsed_pc4 IS NULL
    AND b.listing_point IS NOT NULL
    AND b.latitude BETWEEN p.min_lat AND p.max_lat
    AND b.longitude BETWEEN p.min_lon AND p.max_lon
    AND ST_Contains(p.geom_4326, b.listing_point)
)
SELECT
  airbnb_row_id,
  raw_zipcode,
  COALESCE(parsed_pc4, spatial_pc4) AS pc4,
  pc6,
  CASE
    WHEN COALESCE(parsed_pc4, spatial_pc4) IS NOT NULL THEN TRUE
    ELSE FALSE
  END AS is_valid_zipcode,
  CASE
    WHEN pc6 IS NOT NULL THEN 'VALID_PC6'
    WHEN parsed_pc4 IS NOT NULL THEN 'VALID_PC4'
    WHEN spatial_pc4 IS NOT NULL THEN 'ENRICHED_PC4'
    WHEN raw_zipcode IS NULL OR raw_zipcode = '' THEN 'MISSING'
    ELSE 'INVALID_FORMAT'
  END AS zipcode_status,
  CASE
    WHEN parsed_pc4 IS NOT NULL THEN 'source_zipcode'
    WHEN spatial_pc4 IS NOT NULL THEN 'spatial_backfill'
    ELSE 'unmatched'
  END AS pc4_source,
  latitude,
  longitude,
  room_type,
  accommodates,
  bedrooms,
  price,
  review_scores_value,
  has_recommendation,
  has_review_score,
  _ingested_at
FROM spatially_enriched
WHERE spatial_rn = 1;